In [13]:
import matplotlib

%matplotlib inline 
import warnings; warnings.simplefilter('ignore')    # hide warnings

from matplotlib import pyplot as plt
import numpy as np
import seaborn as sb
from scipy import stats
from scipy.stats import binom

import particles
from particles import distributions as dists
from particles import state_space_models as ssm
from particles.collectors import Moments
from particles import mcmc
from collections import OrderedDict

from epydemix import EpiModel
from epydemix import load_predefined_model
from epydemix.visualization import plot_quantiles, plot_trajectories, plot_posterior_distribution_2d
from epydemix.population import load_epydemix_population, Population, get_available_locations
from epydemix.visualization import plot_contact_matrix, plot_population
from epydemix.utils import compute_simulation_dates
from epydemix import simulate
from epydemix.calibration import ABCSampler, rmse
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
# class DiscreteDist copied from Particles documentation, no changes made except dim is 2 for multivariate
class DiscreteDist:
    dim = 2     # multivariate distribution (default is 1 for univariate)
    dtype = np.int64

    def shape(self, size):
        if size is None:
            return None
        else:
            return (size,) if self.dim == 1 else (size, self.dim)
        
    def logpdf(self, x):
        raise NotImplementedError
    
    def pdf(self, x):
        return np.exp(self.logpdf(x))
    
    def rvs(self, size=None):
        raise NotImplementedError
    
    def ppf(self, u):
        raise NotImplementedError
    

# changed the order of n & p to align with scipy.stats documentation
class NegativeBinomial(DiscreteDist):
    def __init__(self, n=1, p=0.5):
        self.n = n
        self.p = p
    def rvs(self, size = None):
        return np.random.negative_binomial(self.n, self.p, size=size)
    def logpdf(self,x):
        return stats.nbinom.logpmf(x, self.n, self.p) # changed the order of n & p
    def ppf(self, u):
        return stats.nbinom.ppf(u, self.n, self.p) # changed the order of n & p

In [3]:
# Poisson regression to determine the initial state
def Poisson(N, rate):  # where xp = X_{t-1}
    
    chainrule = OrderedDict()
    chainrule['new_inf'] = dists.Dirac(0)
    chainrule['new_rec'] = dists.Dirac(0)
    chainrule['S'] = dists.Dirac(99950)
    chainrule['I'] = dists.Dirac(50)
    chainrule['R'] = dists.Dirac(0)
    return dists.StructDist(chainrule)

# Chain binomial process
def Binomial(xp, beta, gamma, N):  # where xp = X_{t-1}
    # probabilites for new infetions and recoveries
    infection_prob = 1 - np.exp(-beta * xp['I'] / N)
    recovery_prob = 1 - np.exp(-gamma)
    # probability = np.clip(probability, 0, 1)
    
    # Update compartments
    chainrule = OrderedDict()
    chainrule['new_inf'] = dists.Binomial(n = xp['S'].astype(int), p = infection_prob)
    chainrule['new_rec'] = dists.Binomial(n = xp['I'].astype(int), p = recovery_prob)
    chainrule['S'] = dists.Cond(lambda x: dists.Dirac(xp['S'].astype(int) - x['new_inf']))
    chainrule['I'] = dists.Cond(lambda x: dists.Dirac(xp['I'].astype(int) + x['new_inf'] - x['new_rec']))
    chainrule['R'] = dists.Cond(lambda x: dists.Dirac(xp['R'].astype(int) + x['new_rec']))
    return dists.StructDist(chainrule)

In [4]:
# State Space Model

class ChainBinomialModel(ssm.StateSpaceModel):
    default_params = {'N': 100000}
    def PX0(self):                                                      # Initial state of SIR
        return Poisson(100000, 5)
    def PX(self, t, xp):                                                # Hidden Markov process
        return Binomial(xp, self.beta, self.gamma, self.N)
    def PY(self, t, xp, x):                                             # Observation model 
        mu = 0.05 * x['I']
        size = 2
        mu2 = np.maximum(mu, 1e-6)
        p = size/(size + mu2)
        return NegativeBinomial(n = size, p = p)

In [ ]:
def stochastic_sir():
    chain = ChainBinomialModel(beta = 0.3, gamma = 0.1)
    x, y = chain.simulate(100)
    x_s = np.array([xt['S'] for xt in x]).tolist()
    x_s = [xt[0] for xt in x_s]
    x_i = np.array([xt['I'] for xt in x]).tolist()
    x_i = [xt[0] for xt in x_i]
    x_r = np.array([xt['R'] for xt in x]).tolist()
    x_r = [xt[0] for xt in x_r]
    x_inf = np.array([xt['new_inf'] for xt in x]).tolist()
    x_inf = [xt[0] for xt in x_inf]
    return{'S': x_s, 'I': x_i, 'R': x_r, 'incidence': x_inf, 'data': y}

In [23]:
priors = {'beta': stats.uniform(0.1, 0.4),
        'gamma': stats.uniform(0.05, 0.15)}

In [38]:
model = stochastic_sir()

In [65]:
print(model['data'])

[array([3]), array([3]), array([8]), array([12]), array([1]), array([8]), array([25]), array([22]), array([26]), array([9]), array([5]), array([5]), array([87]), array([45]), array([2]), array([32]), array([78]), array([120]), array([59]), array([119]), array([49]), array([38]), array([54]), array([167]), array([247]), array([487]), array([232]), array([101]), array([425]), array([495]), array([304]), array([1675]), array([615]), array([932]), array([1600]), array([1089]), array([184]), array([1600]), array([529]), array([2217]), array([1476]), array([889]), array([1956]), array([1341]), array([1518]), array([1846]), array([630]), array([766]), array([552]), array([1235]), array([801]), array([3321]), array([956]), array([1614]), array([485]), array([1059]), array([983]), array([2058]), array([1439]), array([841]), array([932]), array([302]), array([329]), array([1055]), array([794]), array([1236]), array([1043]), array([166]), array([448]), array([216]), array([38]), array([60]), arra

In [54]:
parameters ={}

In [69]:
def simulate_wrapper():
    results = stochastic_sir()
    return {'data': results['incidence']}

In [70]:
simulate = simulate_wrapper()

In [71]:
print(simulate)

{'data': [0.0, 12.0, 20.0, 15.0, 23.0, 26.0, 32.0, 40.0, 66.0, 58.0, 83.0, 90.0, 116.0, 125.0, 168.0, 201.0, 247.0, 287.0, 364.0, 397.0, 471.0, 556.0, 679.0, 852.0, 950.0, 1123.0, 1280.0, 1541.0, 1784.0, 2031.0, 2330.0, 2648.0, 2854.0, 3136.0, 3460.0, 3761.0, 3958.0, 4094.0, 4187.0, 4102.0, 4042.0, 3938.0, 3727.0, 3573.0, 3326.0, 3002.0, 2803.0, 2522.0, 2277.0, 1936.0, 1754.0, 1522.0, 1343.0, 1229.0, 1074.0, 922.0, 794.0, 729.0, 672.0, 589.0, 494.0, 451.0, 380.0, 346.0, 306.0, 295.0, 271.0, 236.0, 195.0, 180.0, 166.0, 158.0, 139.0, 134.0, 114.0, 97.0, 90.0, 85.0, 77.0, 52.0, 74.0, 60.0, 48.0, 35.0, 43.0, 45.0, 33.0, 41.0, 26.0, 30.0, 23.0, 35.0, 30.0, 20.0, 19.0, 23.0, 16.0, 15.0, 12.0, 12.0]}


In [ ]:
abc_sampler = ABCSampler(simulation_function = simulate_wrapper,
                         priors = priors,
                         parameters = parameters,
                         observed_data = model['data'])

TypeError: 'function' object is not subscriptable

In [57]:
results = abc_sampler.calibrate(strategy="smc", 
                    num_particles=100, 
                    num_generations=5)

Starting ABC-SMC with 100 particles and 5 generations

Generation 1/5 (epsilon: inf)


ValueError: The shapes of observed and simulated data arrays must match.